# ARM97 NetCDF Variable Inventory

This notebook lists variables for both the ARM97 IOP forcing file and the baseline model output file. Each table includes dtype, dimensions, shape, units, and long name.

In [ ]:
from pathlib import Path

import pandas as pd
from netCDF4 import Dataset


def find_file(*candidates: str) -> Path:
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path.resolve()
    raise FileNotFoundError("Could not find file in candidates: " + ", ".join(candidates))


NC_FILES = {
    "IOP forcing": find_file(
        "e3sm_scm_run_scripts_baseline/ARM97_iopfile_4scam.nc",
        "../e3sm_scm_run_scripts_baseline/ARM97_iopfile_4scam.nc",
    ),
    "Model output": find_file(
        "e3sm_scm_run_scripts_baseline/baseline-output/scm_ARM97_baseline/run/case_scripts.eam.h0.1997-06-19-84585.nc",
        "../e3sm_scm_run_scripts_baseline/baseline-output/scm_ARM97_baseline/run/case_scripts.eam.h0.1997-06-19-84585.nc",
    ),
}


def dimensions_table(label: str, nc_file: Path) -> pd.DataFrame:
    with Dataset(nc_file) as ds:
        rows = [
            {
                "file_label": label,
                "dimension": name,
                "size": len(dim),
                "unlimited": dim.isunlimited(),
            }
            for name, dim in ds.dimensions.items()
        ]
    return pd.DataFrame(rows)


def variables_table(label: str, nc_file: Path) -> pd.DataFrame:
    with Dataset(nc_file) as ds:
        rows = []
        for name, variable in ds.variables.items():
            rows.append(
                {
                    "file_label": label,
                    "variable": name,
                    "dtype": str(variable.dtype),
                    "dimensions": ", ".join(variable.dimensions) if variable.dimensions else "scalar",
                    "shape": " x ".join(str(size) for size in variable.shape) if variable.shape else "scalar",
                    "units": getattr(variable, "units", ""),
                    "long_name": getattr(variable, "long_name", ""),
                }
            )
    return pd.DataFrame(rows)


pd.DataFrame(
    [{"file_label": label, "path": str(path)} for label, path in NC_FILES.items()]
)

## Dimensions

In [ ]:
dimensions = pd.concat(
    [dimensions_table(label, path) for label, path in NC_FILES.items()],
    ignore_index=True,
)

dimensions

## IOP Forcing Variables

In [ ]:
iop_variables = variables_table("IOP forcing", NC_FILES["IOP forcing"])
print(f"IOP forcing variables: {len(iop_variables)}")
iop_variables

## Model Output Variables

In [ ]:
model_output_variables = variables_table("Model output", NC_FILES["Model output"])
print(f"Model output variables: {len(model_output_variables)}")
model_output_variables

model_output_variables.to_csv('model_output_variables.csv')

## Combined Variables

In [ ]:
all_variables = pd.concat([iop_variables, model_output_variables], ignore_index=True)
all_variables

## Variable Names Only

In [ ]:
variable_names = {
    "IOP forcing": iop_variables["variable"].tolist(),
    "Model output": model_output_variables["variable"].tolist(),
}

variable_names

## Optional: Save Inventory

In [ ]:
SAVE_CSV = False

if SAVE_CSV:
    output_csv = Path("ARM97_netcdf_variable_inventory.csv")
    all_variables.to_csv(output_csv, index=False)
    output_csv.resolve()
else:
    "Set SAVE_CSV = True and rerun this cell to write ARM97_netcdf_variable_inventory.csv."